# WFPICCS Datathon — Tracheostomy EDA

Shared working notebook (SATI-Q + ANZPICR). Edit any cell — this is a scratchpad, not a final report.

**Data:** three CSVs pulled from the datathon dataset, focused on tracheostomy:
- `FIVARAPA_2015_2025.csv` (SATI-Q, Argentina) — master admissions table, has direct `TRAQ*` fields
- `ANZPICR_DIAG.csv` (ANZPICR, Aus/NZ) — diagnosis/procedure codes, incl. 1506 = tracheostomy
- `ANZPICR_EPI.csv` (ANZPICR) — respiratory support episodes (ETT & trach combined, can't separate)

_Note: ANZPICR has no direct trach flag in the admissions file (`ANZPICR_ADM.csv`), so it's not loaded here._

## 0. Setup — mount Drive, load data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 50)

DATA_DIR = '/content/drive/MyDrive/WFPICCS_Trach_EDA'  # <-- change if you moved the folder

# files are stored gzip-compressed to keep the Drive folder small; pandas reads .gz transparently
fv = pd.read_csv(f'{DATA_DIR}/FIVARAPA_2015_2025.csv.gz', encoding='utf-8-sig', compression='gzip')
diag = pd.read_csv(f'{DATA_DIR}/ANZPICR_DIAG.csv.gz', encoding='utf-8-sig', compression='gzip')
epi = pd.read_csv(f'{DATA_DIR}/ANZPICR_EPI.csv.gz', encoding='utf-8-sig', compression='gzip')

print('FIVARAPA:', fv.shape)
print('ANZPICR_DIAG:', diag.shape)
print('ANZPICR_EPI:', epi.shape)

## 1. SATI-Q (Argentina) — clean & derive trach flags

`TRAQ` = trach used during stay · `TRAQI` = present on admission · `TRAQE` = present on discharge · `TRAQFI`/`TRAQFF` = placement/removal dates.

In [ ]:
def to_flag(s):
    return (s.astype(str).str.strip().str.upper()
            .map({'1': 1, '0': 0, 'SI': 1, 'S': 1, 'N': 0, 'NO': 0, 'TRUE': 1, 'FALSE': 0, 'YES': 1})
            .astype(float))

fv['TRAQ_flag'] = to_flag(fv['TRAQ'])
fv['TRAQI_flag'] = to_flag(fv['TRAQI'])
fv['TRAQE_flag'] = to_flag(fv['TRAQE'])
fv['ARM_flag'] = to_flag(fv['ARM'])

for c in ['FECHAING', 'FECEGR', 'TRAQFI', 'TRAQFF']:
    fv[c + '_dt'] = pd.to_datetime(fv[c], errors='coerce', dayfirst=True)

fv['ADM_YEAR'] = fv['FECHAING_dt'].dt.year
fv['trach_days'] = (fv['TRAQFF_dt'] - fv['TRAQFI_dt']).dt.days

# flag + drop obviously bad duration rows (end date before start date -> data entry error)
fv['trach_days_clean'] = fv['trach_days'].where(fv['trach_days'] >= 0)

n_total = len(fv)
n_trach = (fv['TRAQ_flag'] == 1).sum()
new_trach = ((fv['TRAQI_flag'] == 0) & (fv['TRAQ_flag'] == 1)).sum()

print(f'Total admissions: {n_total:,}')
print(f'Admissions with trach use (TRAQ=1): {n_trach:,} ({n_trach/n_total*100:.2f}%)')
print(f'Trach present on admission: {(fv["TRAQI_flag"]==1).sum():,}')
print(f'Trach present on discharge: {(fv["TRAQE_flag"]==1).sum():,}')
print(f'New trach placed this admission: {new_trach:,}')
print()
print('Trach duration (days), cleaned of negative-date errors:')
print(fv['trach_days_clean'].describe())

### 1a. Yearly trend + trach vs non-trach comparison

In [ ]:
yearly = fv.groupby('ADM_YEAR').agg(admissions=('DNI', 'size'),
                                     trach_admissions=('TRAQ_flag', lambda x: (x == 1).sum()))
yearly['pct_trach'] = (yearly['trach_admissions'] / yearly['admissions'] * 100).round(2)
display(yearly)

grp = fv.groupby(fv['TRAQ_flag'].map({1: 'Trach', 0: 'No trach'}))
print('\nLength of stay (days):')
display(grp['DIAS'].describe())
print('\n% on invasive mechanical ventilation (ARM):')
display((grp['ARM_flag'].mean() * 100).round(1))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

yearly['pct_trach'].plot(ax=axes[0,0], marker='o', color='teal')
axes[0,0].set_title('% of admissions with tracheostomy use, by year')
axes[0,0].set_ylabel('% of admissions'); axes[0,0].grid(alpha=0.3)

fv.boxplot(column='DIAS', by='TRAQ_flag', ax=axes[0,1])
axes[0,1].set_title('PICU length of stay: trach vs non-trach')
plt.suptitle('')

fv.loc[fv['trach_days_clean'].between(0, 200), 'trach_days_clean'].plot(
    kind='hist', bins=40, ax=axes[1,0], color='darkorange', edgecolor='black')
axes[1,0].set_title('Tracheostomy duration (days), 0-200d range')

outcome_ct = pd.crosstab(fv['TRAQ_flag'], fv['RESTRAT'], normalize='index') * 100
outcome_ct.plot(kind='bar', stacked=True, ax=axes[1,1], colormap='tab20')
axes[1,1].set_title('Outcome (RESTRAT) by trach status, %')
axes[1,1].legend(title='RESTRAT', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=7)

plt.tight_layout()
plt.show()

## 2. ANZPICR (Australia/NZ) — trach-related diagnosis codes

No direct trach flag; closest is `ADX` code **1506 = PostOp ENT Tracheostomy** in the diagnosis file, plus related airway codes.

In [ ]:
trach_related_codes = {
    1506: 'PostOp - ENT Tracheostomy',
    434: 'Malacia - Trachea and/or Bronchi',
    436: 'Stenosis - Trachea and/or Bronchi',
    437: 'Tracheo-oesophageal Fistula',
    409: 'Tracheitis',
    1408: 'PostOp Tracheo-oesophageal Fistula Repair',
    1409: 'PostOp Tracheopexy',
    1509: 'PostOp ENT Tracheal Reconstruction / Tracheoplasty',
}

diag['ADX'] = pd.to_numeric(diag['ADX'], errors='coerce')
counts = diag[diag['ADX'].isin(trach_related_codes)]['ADX'].value_counts()
for code_, label in trach_related_codes.items():
    print(f'{code_:>5}  {label:<45} n={counts.get(code_, 0)}')

trach_proc = diag[diag['ADX'] == 1506].copy()
print(f'\nRows with ADX=1506 (tracheostomy procedure): {len(trach_proc)}')
trach_proc.head()

## 3. ANZPICR — respiratory support episodes (context only)

`EPI_CAT` 1 (Invasive Ventilation) and 4 (Intubation) both cover **ETT or tracheostomy** — this file can't tell tracheostomy-specific episodes apart from ETT ones on its own.

In [ ]:
print(epi['EPI_CAT'].value_counts().sort_index())
print('\n1 = Invasive Vent (ETT/Trach), 2 = NIV, 3 = HFNC, 4 = Intubation (ETT/Trach), 5 = ECLS')

## Layman's summary — Argentinian data (SATI-Q, FIVARAPA)

Plain-language version of section 1, for a non-technical read:

- **Out of every 100 kids admitted to these Argentine PICUs, about 5 end up needing a tracheostomy** (a breathing tube placed directly into the windpipe through the neck) at some point during their stay.
- **Most tracheostomies are new, not pre-existing.** Of the roughly 4,400 trach admissions, about 2,000 arrived *without* a trach and got one during the admission — so for those kids it reflects a decision made during their PICU stay, not a pre-existing condition.
- **Kids with a tracheostomy stay in intensive care much longer.** Typical (median) stay is about **16 days**, versus **4 days** for kids without one — roughly 4x longer. The average is skewed even higher (37 vs 8 days) by a smaller group of very long stays.
- **Almost 9 in 10 trach patients (86.5%) were on a breathing machine**, compared with about half (52.6%) of everyone else — consistent with trachs being used for children who need prolonged breathing support.
- **How long the trach stays in, for those with recorded dates:** typical duration is about **9 days** (middle 50% of cases between 4 and 23 days) during the admission itself — many will keep the trach after discharge, so this is PICU-stay duration, not lifetime duration.
- **The rate hasn't trended up or down much over 10 years** — it bounces between about 4% and 6% of admissions each year from 2015 to 2025, no steady rise or fall.
- **Caveat on the timing numbers:** a handful of records have an end date recorded before the start date (a data entry error), which would badly distort an average if not excluded — the stats above use the cleaned version.

## Notes / next steps (edit freely below)

-
-